In [15]:
import io

import requests
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import time

from pandas import Series
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "src/database/BindingDB_All_202605_tsv.zip"


In [16]:
# Goal: get protein sequence + smiles pairs
# task: get pocket patch proteins - need to isolate pocket sequence via UniProt + PDB
# BindingDB: Protein-Ligand affinity data + UniProt IDs
# PDB: will contain information on proteins close to drug (6-8 A)

In [17]:
# Data settings
target_fields = [
    "Ligand SMILES",
    "PDB ID(s) of Target Chain 1",
    "Ki (nM)",  # inhibitor binding strength -> both Ki and Kd are good since we want POCKETS
]

nm_threshold = 100   # isolate high affinity pairs
chunksize = 100000
pdb_resolution = 2.5

In [6]:
chunks = []

with zipfile.ZipFile(DATAPATH) as z_fil:
    tsv_data = [f for f in z_fil.namelist() if f.endswith(".tsv")][0]
    with z_fil.open(tsv_data) as f:
        reader = pd.read_csv(
            f,
            sep="\t",
            dtype=str,
            usecols=target_fields,
            low_memory=False,
            on_bad_lines="skip",
            chunksize=chunksize,
        )
        for chunk in reader:
            filtered = chunk[target_fields]
            chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)

In [18]:
# I only want potent binders
# print(df.shape)  # 3176528, 4

# apply potency filter -> can adjust nm_threshold down the line
df["Ki (nM)"] = pd.to_numeric(df["Ki (nM)"], errors="coerce")
potent_df = df[df["Ki (nM)"] <= nm_threshold]
# print(potent_df.shape)   # 297264, 4

potent_df.head(3)

,Ligand SMILES,PDB ID(s) of Target Chain 1,Ki (nM)
0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCC(O)=...,"1W5Y,1W5X,1W5W,1W5V,2FDE,7UPJ,6UWC,6UWB,6D0E,6...",0.24
1,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(C\C=C\c2cn...,"1W5Y,1W5X,1W5W,1W5V,2FDE,7UPJ,6UWC,6UWB,6D0E,6...",0.25
2,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CC2CC2)C(=...,"1W5Y,1W5X,1W5W,1W5V,2FDE,7UPJ,6UWC,6UWB,6D0E,6...",0.41


In [34]:
unique_smiles_strings = potent_df["Ligand SMILES"].dropna().unique()

all_smiles = set()
for smile in unique_smiles_strings:
    all_smiles.add(smile)

# print(len(all_smiles))  # 155343

In [33]:
unique_pdb_strings = potent_df["PDB ID(s) of Target Chain 1"].dropna().unique()
# print(len(unique_pdb_strings))  # --> 1985 string clusters

all_pdb_ids = set()

for pdb_string in unique_pdb_strings:
    for pid in pdb_string.split(","):
        all_pdb_ids.add(pid.strip())

# print(len(all_pdb_ids)) # --> 30259

In [9]:
def get_pocket(pdb_id):
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    try:
        req = requests.get(url, timeout=10)
        if req.status_code != 200:
            return None
        data = req.json()

        req_data = data.get("rcsb_entry_info", {})

        return {
            "pdb_id": pdb_id.upper(),
            "resolution": req_data.get("resolution_combined", [999])[0],
            "ligand_count": req_data.get("nonpolymer_entity_count", 0),
        }

    except Exception as e:
        print(f"Couldn't get info on {pdb_id} : {e}")
        return None


def pdb_filters(all_pdb_ids, delay=0.1):
    result_df = []
    for pdb_id in tqdm(all_pdb_ids, total=len(all_pdb_ids), desc="Validating PDBs + Building Cache"):
        meta_data = get_pocket(pdb_id)

        # apply filters -> want relatively high resolution and a sequence that's acc bound to smth
        if meta_data is None:
            continue

        time.sleep(delay)

        if meta_data["resolution"] > 2.5:
            continue
        if meta_data["ligand_count"] == 0:
            continue

        result_df.append({
            "PDB ID": pdb_id,
        })

    return pd.DataFrame(result_df)

pdb_df = pdb_filters(all_pdb_ids=all_pdb_ids, delay=0.1)
pdb_df.to_csv("unique_pdb_cache.csv")

Validating PDBs + Building Cache:  44%|████▍     | 13398/30259 [1:02:21<16:01:50,  3.42s/it]

Couldn't get info on 3I6M : HTTPSConnectionPool(host='data.rcsb.org', port=443): Read timed out. (read timeout=10)


Validating PDBs + Building Cache: 100%|██████████| 30259/30259 [2:20:49<00:00,  3.58it/s]   


In [31]:
# Ok so pdb_df is gonna contain all valid unique pdb IDs
# potent_df is gonna contain all SMILES
# going to need to switch to mmCIF but that's fine - same 4 char codes
# isolate atoms within 8 angstroms of ligand - look for LIG comp_id

# lemme just place this here to get the functions defined
# will call on it using thread in another cell

# notes from biopython cookbook
# structure consists of models
# model consists of chains
# chain consists of residues
# residue consists of atoms

from concurrent.futures import ThreadPoolExecutor, as_completed

from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB import Selection
from Bio.PDB.Polypeptide import PPBuilder

# load all unique pdb ids as a list
pdb_df = pd.read_csv("unique_pdb_cache.csv")
pdb_list = pdb_df["PDB ID"].to_list()

# Residues to exclude: waters, common ions, solvents
EXCLUDE_HETS = {
    "HOH", "WAT", "DOD",  # water
    "NA", "K", "MG", "CA", "ZN", "FE", "MN", "CU", "CL", "BR", "I",  # ions
    "SO4", "PO4", "GOL", "EDO", "PEG", "MPD", "DMS", "ACT", "ACE",   # solvents/buffers
    "NO3", "CO3", "EPE", "MES", "TRS", "HEZ"
}

AA_codetoletter = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    "MSE": "selM",  # selenomethionine: gonna need to do some literature on these later.
    "SEP": "phoS",  # phosphoserine
    "TPO": "phoT",  # phosphothreonine
    "PTR": "phoY",  # phosphotyrosine
    "HYP": "hydP",  # hydroxyproline
}

ANGSTROM_THRESHOLD = 8.0


def fetch_mmcif_stream(pdb_id: str) -> io.StringIO | None:
    """I want to avoid disk writing, it's like 20k entries"""
    url = f"https://files.rcsb.org/download/{pdb_id.lower()}.cif"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return None
    return io.StringIO(r.text)

def get_ligand_atoms(struct) -> list:
    """Extract HETATM belonging to drug-like ligands"""
    ligand_atoms = []
    for model in struct:
        residues = model.get_residues()
        for residue in residues:
            if residue.id[0].startswith("H_"):
                resname = residue.resname.strip()
                if resname not in EXCLUDE_HETS:
                    for atom in residue.get_atoms():
                        if atom.element != "H":
                            ligand_atoms.append((atom, residue))

    return ligand_atoms


def get_ligand_data(ligand_atoms):
    """get the geometric center (avg. pos) of all atoms within the ligand mol"""
    coords = np.array([atom.get_vector().get_array()
                       for atom, _ in ligand_atoms])
    return coords, coords.mean(axis=0)

def get_pocket_residues(struct, ligand_atoms, threshold: float = ANGSTROM_THRESHOLD):
    """
    obtain concatted 1-letter codes of all protein residues within 8 Angstroms (default) of any ligand atom
    :param threshold:
    :return:
    """
    if not ligand_atoms:
        return []

    ligand_coords, ligand_centroid = get_ligand_data(ligand_atoms)

    binding_site = []

    for model in struct:
        for residue in model.get_residues:
            if residue.id[0] != " ":  # blank for standard amino and nucleic acids
                continue



